In [ ]:
# Cell 1: Setup
import sys
sys.path.append('..')

from src.rag_chain import NetflixGPTWithMemory
from src.conversation_memory import ConversationMemory

print("Initializing Netflix GPT with Memory...")
rag = NetflixGPTWithMemory(
    model_name="llama3.2",
    max_memory_turns=5,
    session_id="notebook_test"
)
print("✅ Ready!\n")

Initializing Netflix GPT with Memory...


NameError: name 'NetflixGPTWithMemory' is not defined

In [ ]:
# Cell 2: Test Basic Memory
print("="*70)
print("TEST 1: Basic Conversation Flow")
print("="*70 + "\n")

response1 = rag.query_with_memory("Recommend action movies")
print(f"Q1: {response1['question']}")
print(f"A1: {response1['answer'][:200]}...")
print(f"Sources: {[s['title'] for s in response1['sources'][:3]]}\n")

response2 = rag.query_with_memory("Tell me more about the first one")
print(f"Q2: {response2['question']}")
print(f"Follow-up detected: {response2['is_followup']}")
print(f"A2: {response2['answer'][:200]}...")

In [ ]:
# Cell 3: Test Preference Tracking
print("\n" + "="*70)
print("TEST 2: Preference Tracking")
print("="*70 + "\n")

rag.clear_memory()

queries = [
    "I love sci-fi movies with deep plots",
    "But I don't like horror or scary stuff",
    "Something from the 90s would be great"
]

for query in queries:
    response = rag.query_with_memory(query)
    print(f"Q: {query}")
    print(f"A: {response['answer'][:150]}...\n")

print("Detected Preferences:")
summary = rag.get_memory_summary()
print(f"  {summary['preferences']}")

In [ ]:
# Cell 4: Test Follow-up Detection
print("\n" + "="*70)
print("TEST 3: Follow-up Question Detection")
print("="*70 + "\n")

rag.clear_memory()

# Initial query
response1 = rag.query_with_memory("What are good thriller movies?")
print(f"Q1: What are good thriller movies?")
print(f"   Is follow-up: {response1['is_followup']}\n")

# Various follow-up styles
followups = [
    "Tell me more about that",
    "What about the second one?",
    "Are there more like those?",
    "Actually, show me comedies instead"  # Topic change
]

for query in followups:
    response = rag.query_with_memory(query)
    print(f"Q: {query}")
    print(f"   Is follow-up: {response['is_followup']}")

In [ ]:
# Cell 5: Test Memory Window
print("\n" + "="*70)
print("TEST 4: Memory Window (Max 5 turns)")
print("="*70 + "\n")

rag.clear_memory()

# Add 7 turns (should keep only last 5)
for i in range(7):
    query = f"Recommend movie type {i+1}"
    response = rag.query_with_memory(query)
    print(f"Turn {i+1}: Turns in memory = {len(rag.memory.history)}")

print(f"\nFinal memory size: {len(rag.memory.history)} turns")
print("(Should be 5 - the maximum)")

In [ ]:
# Cell 6: Test Mentioned Movies Tracking
print("\n" + "="*70)
print("TEST 5: Tracking Mentioned Movies")
print("="*70 + "\n")

rag.clear_memory()

response1 = rag.query_with_memory("Best action movies")
response2 = rag.query_with_memory("How about thriller movies?")
response3 = rag.query_with_memory("Any good comedies?")

recent_movies = rag.memory.get_recent_movies(n=10)
print(f"Movies mentioned in conversation ({len(recent_movies)} total):")
for i, movie in enumerate(recent_movies, 1):
    print(f"  {i}. {movie}")

In [ ]:
# Cell 7: Test Save and Load
print("\n" + "="*70)
print("TEST 6: Save and Load Conversation")
print("="*70 + "\n")

# Current state
print("Before save:")
print(f"  Turns: {rag.memory.turn_count}")
print(f"  Movies: {len(rag.memory.mentioned_movies)}")

# Save
rag.save_conversation("../data/conversations/notebook_test.json")

# Create new instance
print("\nCreating new RAG instance...")
rag_new = NetflixGPTWithMemory(
    model_name="llama3.2",
    session_id="notebook_test"
)

# Load previous conversation
rag_new.load_conversation("../data/conversations/notebook_test.json")

print("\nAfter load:")
print(f"  Turns: {rag_new.memory.turn_count}")
print(f"  Movies: {len(rag_new.memory.mentioned_movies)}")
print("  ✅ Conversation preserved!")

In [ ]:
# Cell 8: Test Context Enhancement
print("\n" + "="*70)
print("TEST 7: Query Enhancement with Context")
print("="*70 + "\n")

rag.clear_memory()

# First query
response1 = rag.query_with_memory("Recommend The Matrix")
print("Initial query processed\n")

# Test enhancement
test_queries = [
    "Tell me more about it",
    "What about similar movies?",
    "That sounds interesting"
]

for query in test_queries:
    enhanced = rag.memory.enhance_query_with_context(query)
    print(f"Original:  {query}")
    print(f"Enhanced:  {enhanced}")
    print()

In [ ]:
# Cell 9: Full Conversation Simulation
print("\n" + "="*70)
print("TEST 8: Natural Conversation Flow")
print("="*70 + "\n")

rag.clear_memory()

conversation = [
    "I'm looking for something exciting to watch tonight",
    "I love those! Tell me more about the first one",
    "Actually, I've already seen that. What else?",
    "Perfect! I'll watch that one. Thanks!"
]

for i, query in enumerate(conversation, 1):
    print(f"\n--- Turn {i} ---")
    response = rag.query_with_memory(query)
    
    print(f"User: {query}")
    print(f"Is follow-up: {response['is_followup']}")
    print(f"Assistant: {response['answer'][:150]}...")
    
    if response.get('sources'):
        print(f"Mentioned: {[s['title'] for s in response['sources'][:2]]}")

# Final summary
print("\n" + "="*70)
print("Conversation Summary:")
summary = rag.get_memory_summary()
for key, value in summary.items():
    print(f"  {key}: {value}")
print("="*70)

# Cell 10: Cleanup
print("\n🧹 Cleanup: Clearing memory for next test")
rag.clear_memory()
print("✅ Ready for new conversation")